In [5]:
# Import the usual libraries and variables.
%run standard_imports.py

# Install a pip package in the current Jupyter kernel.
import sys
!{sys.executable} -m pip install --user pypika

DEPRECATION: Python 2.7 will reach the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 won't be maintained after that date. A future version of pip will drop support for Python 2.7.


In [6]:
from datetime import date
from pypika import MySQLQuery, Table, Order, functions as fn

start_date = date(2018, 1, 1)
end_date = date(2019, 1, 1)

db = Db("~/.clover/p801.cfg")

charge = Table("charge")
merchant_app_charge = Table("merchant_app_charge")
merchant_app = Table("merchant_app")
developer_app = Table("developer_app")
developer = Table("developer")

gross_revenue = fn.Sum(charge.amount).as_("gross_revenue")
gross_developer_portion = fn.Sum(fn.Coalesce(charge.developer_portion,
                                             charge.amount * 0.7)).as_("gross_developer_portion")
gross_clover_portion = fn.Sum(charge.amount - fn.Coalesce(charge.developer_portion,
                                                          charge.amount * 0.7)).as_("gross_clover_portion")

q = MySQLQuery.from_(charge).join(
    merchant_app_charge
).on(
    merchant_app_charge.charge_id == charge.id 
).join(
    merchant_app
).on(
    merchant_app.id == merchant_app_charge.merchant_app_id
).join(
    developer_app
).on(
    developer_app.id == merchant_app.app_id
).join(
    developer
).on(
    developer.id == developer_app.developer_id
).select(
    developer.uuid,
    developer.name,
    gross_revenue,
    gross_developer_portion,
    gross_clover_portion
).where(
    (developer.name != "Clover") &
    (developer.is_rev_share_flat_rate == 0)
).where(
    (charge.system_type == "INFOLEASE") &
    (charge.status.isin(["DISBURSED", "BILLED"])) &
    (charge.currency == "USD")
).where(
    charge.created_time[start_date:end_date]
).groupby(developer.id) \
.orderby(gross_clover_portion, order=Order.desc)

query = q.get_sql()
print(query)

df = pd.read_sql(query, con=db.conn)
db.close()

df

SELECT `developer`.`uuid`,`developer`.`name`,SUM(`charge`.`amount`) `gross_revenue`,SUM(COALESCE(`charge`.`developer_portion`,`charge`.`amount`*0.7)) `gross_developer_portion`,SUM(`charge`.`amount`-COALESCE(`charge`.`developer_portion`,`charge`.`amount`*0.7)) `gross_clover_portion` FROM `charge` JOIN `merchant_app_charge` ON `merchant_app_charge`.`charge_id`=`charge`.`id` JOIN `merchant_app` ON `merchant_app`.`id`=`merchant_app_charge`.`merchant_app_id` JOIN `developer_app` ON `developer_app`.`id`=`merchant_app`.`app_id` JOIN `developer` ON `developer`.`id`=`developer_app`.`developer_id` WHERE `developer`.`name`<>'Clover' AND `developer`.`is_rev_share_flat_rate`=0 AND `charge`.`system_type`='INFOLEASE' AND `charge`.`status` IN ('DISBURSED','BILLED') AND `charge`.`currency`='USD' AND `charge`.`created_time` BETWEEN '2018-01-01' AND '2019-01-01' GROUP BY `developer`.`id` ORDER BY `gross_clover_portion` DESC


,uuid,name,gross_revenue,gross_developer_portion,gross_clover_portion
0,2SW5NA5C30G10,Homebase,147940634.0,103591016.0,44349618.0
1,3GVN1P855JXEA,Abreeze Technology,98354490.0,68859807.3,29494682.7
2,DK22NG4RNCM7W,Seven Spaces,86335814.0,60459697.6,25876116.4
3,R9XNECD2A8VKP,Infuse,76542947.0,53586075.0,22956872.0
4,QCRD5T0ZX1HSG,DAVO Technologies,47875614.0,33516747.0,14358867.0
5,DJJ1DMZS8AP42,Menufy,41391875.0,28975744.0,12416131.0
6,H0MPC0J3SVFY6,4 Leaf Labs,37437668.0,26208952.0,11228716.0
7,1Z9K2PP9WGRKT,"Shopventory, Inc.",34283259.0,23998439.0,10284820.0
8,FXP6KXWMEVCB4,Zaytech,31073362.0,21754166.0,9319196.0
9,5CQY9KWKPC76M,"Appheaven, LLC",25741278.0,18025076.0,7716202.0
